# ISTAT TRAFFIC DATA ANALYSIS

First, let's import all the needed packages

In [1]:
# import packages
import os
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# set style for plots
sns.set_style("whitegrid")

### UNDERSTANDING THE DATA
First let's retrieve data from the `.csv` file obtained by fetching the API

In [2]:
def get_api(save_path: str, url: str, header: dict = None) -> None:
    """
    Fetches data from the specified API endpoint and saves it to a CSV file.
    
    Parameters
    ----------
    url (str):                  The API endpoint URL.
    header (dict, optional):    Headers for the API request.
    """

    try:
        # GET request 
        print("Fetching data from the url...")
        r = requests.get(url, headers=header)
        
        # Check request status
        r.raise_for_status()

        filename = save_path
        with open(filename, "wb") as file:
            file.write(r.content)

    except requests.exceptions.HTTPError as err:
        print(f"HTTP error occurred: {err}")
    except Exception as err:
        print(f"An error occurred: {err}")

In [3]:
data = "data/istat_traffic_accidents.csv"

url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

if not os.path.exists(data):
    # run the function to retrieve the data from the API
    get_api(save_path=data, url=url, header=header)

# Creating pandas DataFrame
df_raw = pd.read_csv(data)

print(df_raw.info())
df_raw.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  object 
 1   FREQ              573552 non-null  object 
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  object 
 4   RESULT            573552 non-null  object 
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
80983,IT1:41_983(1.0),A,8025,KILLINJ,F,2002,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
285060,IT1:41_983(1.0),A,37048,ROADACC,9,2001,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142302,IT1:41_983(1.0),A,16122,ROADACC,9,2006,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
258058,IT1:41_983(1.0),A,30021,KILLINJ,M,2020,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
549642,IT1:41_983(1.0),A,98058,KILLINJ,F,2013,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
291335,IT1:41_983(1.0),A,41003,ROADACC,9,2015,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
255160,IT1:41_983(1.0),A,29032,KILLINJ,M,2020,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67326,IT1:41_983(1.0),A,6096,KILLINJ,M,2011,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405499,IT1:41_983(1.0),A,66092,KILLINJ,M,2012,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204231,IT1:41_983(1.0),A,22057,KILLINJ,F,2004,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


At first glance, we can see that we only have information for the first seven columns. Meanwhile, the others have all NaN values, meaning we don't have information for them yet.  
Now, let's take a closer look at the columns with data, such as the time period range.

In [4]:
all_df_columns = df_raw.columns

# selecting only columns with no-null values
df = df_raw[all_df_columns[:7]]

# see all the unique values in each column
for col in df.columns:
    u = df[col].unique()
    print(f"Unique values in {col}: {len(u)}")
    print(u)
    print("---")

Unique values in DATAFLOW: 1
['IT1:41_983(1.0)']
---
Unique values in FREQ: 1
['A']
---
Unique values in REF_AREA: 8578
[  1001   1002   1003 ... 111105 111106 111107]
---
Unique values in DATA_TYPE: 2
['KILLINJ' 'ROADACC']
---
Unique values in RESULT: 3
['F' 'M' '9']
---
Unique values in TIME_PERIOD: 24
[2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014
 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
---
Unique values in OBS_VALUE: 1830
[  10    7   13 ... 1081  880  969]
---


The first information we can gather is that our data range is 24 years, from 2001 to 2024. In the dataset, there are two types of accidents: "KILLINJ" and "ROADACC." From the tags, we can hypothesize that "KILLINJ" is a fatal or a serious accident and "ROADACC" is a simple road accident. However, we still have to understand what the other columns represent.  
- DATAFLOW: the possible code that represents the methodology of how the data are inserted into the dataset. Since we have only one unique value, it could explain why all the records were registered in the same way. This is also not useful for our analysis.
- FREQ: It could possibly represent the frequency, but we only have one unique value, "A." We have to investigate what this code means.
- REF_AREA: A code that represents the location or city where the incident occurred.
- RESULT: For this, we only have three distinct values: two letters and a number. More in depth information is required.
- OBS_VALUES: It could represent the total number of incidents that occurred in that specific area and time.
   

After searching the internet we found information about the structure and meaning of the dataset.  
From the official ISTAT website (https://www.istat.it/classificazioni-e-strumenti/web-services-sdmx/), we learned that the data in the APIs follows the SDMX standard protocol. Therefore, we were able to access the glossary for this structure on the official website, where we could download the universal glossary (https://sdmx.org/wp-content/uploads/SDMx_Glossary_version_2_1-Final-2.docx).   
Now that we have this information, we can describe each column of the data frame to help us continue the analysis.
- **DATAFLOW:** `41_983` represents the unique code that ISTAT uses to indicate the SDMX data flow *Incidenti, Morti e Feriti - Comuni* where the prefix *IT1* indicates Italy and *(1.0)* could indicate the version. Information obtained from the website https://datiistat.it/alldataflows#
- **FREQ:** rappresents the *"time interval at which observations occur over a given time period."* In this casa the letter A indicates a annual period (A = anno).
- **REF_AREA:** rappresents the *"Country or geographic area to which the measured statistical phenomenon relates"*. In this case each value indicates a city (comune)
- **DATA_TYPE:** This is not included in the glossary, but given the context, we can safely assume that "KILLINJ" represents a fatal or sirious accident and "ROADACC" represents a simple road accident.
- **RESULT:** no informtion found yet.
- **TIME_PERIOD:** rappresents the *"Timespan or point in time to which the observation actually refers"*. In this case each value indicates a the year
- **OBS_VALUE:** rappresents the *"Value of a particular variable"*. In this case, each value represents the total number of accidents that occurred in a specific city and year.
  
Since we don't have information for the `RESULT` entry only, let's investigate the data frame to understand its meaning.

In [5]:
print(df.loc[df["DATA_TYPE"] == "KILLINJ", "RESULT"].unique())
print(df.loc[df["DATA_TYPE"] == "ROADACC", "RESULT"].unique())

['F' 'M']
['9']


Here, we see how the three unique values in RESULT are specifically associated. The letters F and M are present only for the record with DATA_TYPE = KILLINJ, while the number 9 is present only when associated with DATA_TYPE = ROADACC.  
In the context of KILLINJ, where we mention how it represents cases of fatalities or injuries (the code is an aggregation of killed and injured), the letters F and M stand for FERITI and MORTI.  
After finding no information about the meaning of the number 9, we searched with AI tools and found that the number represents a specific code indicating "total." In simple terms, that record represents the "total" number of incidents, whether fatal or injury-related, without taking other information into account.  

So the updated legend of the dataset is the following:
- **DATAFLOW:** `41_983` represents the unique code that ISTAT uses to indicate the SDMX data flow *Incidenti, Morti e Feriti - Comuni* where the prefix *IT1* indicates Italy and *(1.0)* could indicate the version.
- **FREQ:** rappresents the *"time interval at which observations occur over a given time period."* In this casa the letter A indicates a annual period (A = anno).
- **REF_AREA:** rappresents the *"Country or geographic area to which the measured statistical phenomenon relates"*. In this case each value indicates a city (comune)
- **DATA_TYPE:** This is not included in the glossary, but given the context, we can safely assume that "KILLINJ" represents a fatal or sirious accident and "ROADACC" represents a simple road accident.
- **RESULT:** type of accident that occurred. It distinguishes between fatal (M) and injury (F), while "9" only represents the total number without any other information.
- **TIME_PERIOD:** rappresents the *"Timespan or point in time to which the observation actually refers"*. In this case each value indicates a the year
- **OBS_VALUE:** rappresents the *"Value of a particular variable"*. In this case, each value represents the total number of accidents that occurred in a specific city and year.

### COMPLETING THE DATA
From the INSTAT data, we only have the numeric code for the city (REF_AREA), but we don't know which city it indicates. To address this issue, we will use another SITUAS dataset that provides information about the numeric code, the corresponding city, and its population. This information is useful for calculating per capita statistical results.  
We downloaded the CSV file from https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2025-01-01, making sure to select January 1st of each year so that we have the population data for each previous year. This could be useful, as we could see how the proportion of accidents to the population changes over time.  

From these datasets, we only need information about the identification numeric code and the population size for each year. To accomplish this, we implemented a function that creates a smaller data frame containing all the necessary information.

In [6]:
def create_cities_df(path: str, features: list) -> None:
    """
    Create a combined dataframe for all cities across multiple years.

    Parameters
    ----------
    path : str          The path to the directory containing the CSV files.
    features : list     The list of features to include in the combined dataframe.
    """

    year_list = np.arange(2001, 2025).astype('int32')

    df_temp_prev = pd.read_csv(path+str(year_list[0])+".csv", sep=";")
    df_temp_prev = df_temp_prev[features]

    for i in range(1, len(year_list)):
        df_temp = pd.read_csv(path+str(year_list[i])+".csv", sep=";")
        df_temp = df_temp[features]

        # concatenate the dataframes
        df_temp_prev = pd.concat([df_temp_prev, df_temp], ignore_index=True)
    
    # sorting by city code and year
    df_temp_prev = df_temp_prev.sort_values(by=['Codice Comune (numerico)', 'Anno (Popolazione residente)']).reset_index(drop=True)

    # save the new dataframe
    df_temp_prev.to_csv("data/cities_data.csv", index=False)


In [7]:
cities_data = "data/situas/Comuni - Dimensione Data Indagine 31-12-"
cities_features = ["Codice Comune (numerico)", "Comune", "Popolazione residente", "Anno (Popolazione residente)"]

if not os.path.exists(cities_data):
    # run the function to retrieve the data from the API
    create_cities_df(path=cities_data, features=cities_features)

# Creating pandas DataFrame
cities_data_clean = "data/cities_data.csv"
df_cities = pd.read_csv(cities_data_clean)

print(df_cities.info())
df_cities.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192731 entries, 0 to 192730
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Codice Comune (numerico)      192731 non-null  int64 
 1   Comune                        192707 non-null  object
 2   Popolazione residente         192731 non-null  int64 
 3   Anno (Popolazione residente)  192731 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 5.9+ MB
None


,Codice Comune (numerico),Comune,Popolazione residente,Anno (Popolazione residente)
127571,64016,Candida,1123,2008
139145,69021,Celenza sul Trigno,1080,2002
34191,12074,Gemonio,2644,2002
42041,15042,Calvignasco,1194,2013
124508,62058,San Giorgio del Sannio,10023,2017
148393,75064,Ruffano,9756,2014
23761,6138,Pozzolo Formigaro,4723,2015
99060,41066,Urbania,6655,2001
152343,77031,Scanzano Jonico,7528,2022
111540,55033,Avigliano Umbro,2585,2009


From the `.info()` we can see that there are some `NaN` values for the *Comune* columns. Let's investigate.

In [8]:
df_cities[df_cities["Comune"].isna()]

,Codice Comune (numerico),Comune,Popolazione residente,Anno (Popolazione residente)
3990,1168,NaN,7749,2001
3991,1168,NaN,7777,2002
3992,1168,NaN,7822,2003
3993,1168,NaN,7838,2004
3994,1168,NaN,7815,2005
3995,1168,NaN,7798,2006
3996,1168,NaN,7866,2007
3997,1168,NaN,7873,2008
3998,1168,NaN,7859,2009
3999,1168,NaN,7986,2010


**Summing Check:** If we subtract one array from the other and sum all the values, we can see if they are equal. If the total equals zero, then both arrays are identical; otherwise, they are different.

In [9]:
if np.sum(np.abs(df["REF_AREA"].unique() - df_cities["Codice Comune (numerico)"].unique())) == 0:
    print("The city codes are identical.")
else:
    print("The city codes are different.")

The city codes are identical.


We can perform a quick check on the website (https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2024-01-01) by filtering for the city code 1168 in any year and see that the name is actually **None**. Pandas misinterpreted the name as the variable `None` which is why we get a `NaN` value when we print the data frame.  
You can check if a city named **None** actually exists by looking at this list.: https://www.comuni-italiani.it/001/lista.html  
We can fix this issue in a very direct and simple way.

In [10]:
# fix missing values
df_cities.loc[df_cities['Comune'].isnull(), 'Comune'] = 'None'

print(df_cities.info())
df_cities.loc[df_cities['Comune'] == 'None']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192731 entries, 0 to 192730
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Codice Comune (numerico)      192731 non-null  int64 
 1   Comune                        192731 non-null  object
 2   Popolazione residente         192731 non-null  int64 
 3   Anno (Popolazione residente)  192731 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 5.9+ MB
None


,Codice Comune (numerico),Comune,Popolazione residente,Anno (Popolazione residente)
3990,1168,None,7749,2001
3991,1168,None,7777,2002
3992,1168,None,7822,2003
3993,1168,None,7838,2004
3994,1168,None,7815,2005
3995,1168,None,7798,2006
3996,1168,None,7866,2007
3997,1168,None,7873,2008
3998,1168,None,7859,2009
3999,1168,None,7986,2010


Now that the dataset has been fixed, we can save the corrected CSV file for the cities.

In [11]:
# rename columns for better reading
new_columns_name = ['Codice Comune', 'Comune', 'Residenti', 'Anno']
df_cities.columns = new_columns_name

df_cities.to_csv("data/cities_data_fixed.csv", index=False)

### JOIN DATASET
Now that we have prepared the two datasets, we can join them. This will provide information about the city where the data is from and the number of residents for each record of a traffic accident.  
To do this, we will join the two dataframes using the city code and the year. For each city code in the INSTA `df`, there are multiple records because we are covering a time range of 24 years.  

But let's do before some other data checking.

In [12]:
# example
df[df['REF_AREA'] == 24127]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
230868,IT1:41_983(1.0),A,24127,KILLINJ,F,2019,13
230869,IT1:41_983(1.0),A,24127,KILLINJ,F,2020,14
230870,IT1:41_983(1.0),A,24127,KILLINJ,F,2021,13
230871,IT1:41_983(1.0),A,24127,KILLINJ,F,2022,20
230872,IT1:41_983(1.0),A,24127,KILLINJ,F,2023,18
230873,IT1:41_983(1.0),A,24127,KILLINJ,F,2024,11
230874,IT1:41_983(1.0),A,24127,KILLINJ,M,2019,0
230875,IT1:41_983(1.0),A,24127,KILLINJ,M,2020,1
230876,IT1:41_983(1.0),A,24127,KILLINJ,M,2021,0
230877,IT1:41_983(1.0),A,24127,KILLINJ,M,2022,1


For example, the city of Lusiana Conco's values are covered from 2019 to 2024, and we can see that we have a record for each data type (F, M, 9) for each year in the data. Let's verify this for each city.

In [13]:
# I know, it's a very ugly way to do it... but it works

city_list = df['REF_AREA'].unique()

particular_city_code = []
for code in city_list:
    temp_df = df[df['REF_AREA'] == code]
    if temp_df.shape[0] % 3 != 0:
        particular_city_code.append(code)
        continue
        
    elif (temp_df[temp_df['RESULT'] == 'F'].shape[0] != temp_df[temp_df['RESULT'] == 'M'].shape[0]) or (temp_df[temp_df['RESULT'] == 'M'].shape[0] != temp_df[temp_df['RESULT'] == '9'].shape[0]):
        particular_city_code.append(code)
        continue

    else:
        continue

if len(particular_city_code) == 0:
    print("Hypothesis supported")

print(f'Number of particular cities: {len(particular_city_code)}')


Hypothesis supported
Number of particular cities: 0


As we expected, we have the same number of values for each city for the three data types. This helps us better understand the structure of the data.  
For each city code and year, we have the respective number of fatal accidents (M), serious accidents (F), and simple collisions (9).  

**Now let's join the two dataframe:**  
We want a single data frame that includes the corresponding city name and population for each record of the ISTAT data frame.

In [22]:
data_istat_situa = "data/istat_situa_traffic_accidents_data.csv"

if not os.path.exists(data_istat_situa):
    # create the combined dataframe
    df_combined = pd.merge(
        df,
        df_cities,
        left_on=['REF_AREA', 'TIME_PERIOD'],
        right_on=['Codice Comune', 'Anno'],
        how='left'
    )

    # Drop the duplicate columns
    df_combined = df_combined.drop(columns=['Codice Comune', 'Anno'])

else:
    df_combined = pd.read_csv(data_istat_situa)

print(f'instat dataframe shape: {df.shape[0]}')
print(f'instat cities dataframe shape: {df_cities.shape[0]}')
print(f'instat + situa dataframe shape: {df_combined.shape[0]}')

print(df_combined.info())
df_combined.sample(10)

instat dataframe shape: 573552
instat cities dataframe shape: 192731
instat + situa dataframe shape: 573552
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATAFLOW     573552 non-null  object 
 1   FREQ         573552 non-null  object 
 2   REF_AREA     573552 non-null  int64  
 3   DATA_TYPE    573552 non-null  object 
 4   RESULT       573552 non-null  object 
 5   TIME_PERIOD  573552 non-null  int64  
 6   OBS_VALUE    573552 non-null  int64  
 7   Comune       572700 non-null  object 
 8   Residenti    572700 non-null  float64
dtypes: float64(1), int64(3), object(5)
memory usage: 39.4+ MB
None


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Comune,Residenti
403749,IT1:41_983(1.0),A,66067,ROADACC,9,2021,0,Pereto,638.0
169949,IT1:41_983(1.0),A,18057,KILLINJ,M,2018,0,Corvino San Quirico,1017.0
319753,IT1:41_983(1.0),A,51014,ROADACC,9,2017,1,Chitignano,910.0
569311,IT1:41_983(1.0),A,109012,KILLINJ,F,2011,0,Monsampietro Morico,695.0
530353,IT1:41_983(1.0),A,95039,KILLINJ,M,2018,0,Palmas Arborea,1537.0
188818,IT1:41_983(1.0),A,20018,ROADACC,9,2005,9,Cavriana,3818.0
267502,IT1:41_983(1.0),A,31017,KILLINJ,M,2005,0,Sagrado,2194.0
247671,IT1:41_983(1.0),A,28031,ROADACC,9,2019,5,Cinto Euganeo,1977.0
88898,IT1:41_983(1.0),A,9069,KILLINJ,F,2003,0,Zuccarello,306.0
158693,IT1:41_983(1.0),A,17104,KILLINJ,M,2006,1,Marcheno,4353.0


Using the `.info()` function, we can see some city codes are missing from the SITUAS dataset. Let's identify them. 

In [23]:
df_combined[df_combined['Comune'].isnull()]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Comune,Residenti
65103,IT1:41_983(1.0),A,6064,KILLINJ,F,2019,0,NaN,NaN
65122,IT1:41_983(1.0),A,6064,KILLINJ,M,2019,0,NaN,NaN
65141,IT1:41_983(1.0),A,6064,ROADACC,9,2019,0,NaN,NaN
66828,IT1:41_983(1.0),A,6089,KILLINJ,F,2019,0,NaN,NaN
66847,IT1:41_983(1.0),A,6089,KILLINJ,M,2019,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
566295,IT1:41_983(1.0),A,107022,KILLINJ,M,2016,0,NaN,NaN
566306,IT1:41_983(1.0),A,107022,ROADACC,9,2016,3,NaN,NaN
566317,IT1:41_983(1.0),A,107023,KILLINJ,F,2016,0,NaN,NaN
566328,IT1:41_983(1.0),A,107023,KILLINJ,M,2016,0,NaN,NaN


By looking at the .csv file for city data and checking directly on the website where we downloaded the data, we can see that we don't have data for some cities that covers the same time range as the INSTAT data.  
For example, the city with the code equal to 6064 (Cuccaro Monferrato, TO) has no data for the year 2019. With a quick search on Wikipedia (https://it.wikipedia.org/wiki/Cuccaro_Monferrato), we can read that, as of January 1, 2019, the city will join with Lu (city code 6089) to form a new city called Lu e Cuccaro Monferrato, which will have a different city code (6193).

<img src="images/6064_example.png" width="600">
<img src="images/Lu_example.png" width="600">

In this case, we can delete the 2019 records for both cities since they appear to be joined with the new city code from the same year. Let's check if there are other similar cases in the entire data frame.

In [24]:
df_combined[df_combined['Comune'].isnull()]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Comune,Residenti
65103,IT1:41_983(1.0),A,6064,KILLINJ,F,2019,0,NaN,NaN
65122,IT1:41_983(1.0),A,6064,KILLINJ,M,2019,0,NaN,NaN
65141,IT1:41_983(1.0),A,6064,ROADACC,9,2019,0,NaN,NaN
66828,IT1:41_983(1.0),A,6089,KILLINJ,F,2019,0,NaN,NaN
66847,IT1:41_983(1.0),A,6089,KILLINJ,M,2019,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
566295,IT1:41_983(1.0),A,107022,KILLINJ,M,2016,0,NaN,NaN
566306,IT1:41_983(1.0),A,107022,ROADACC,9,2016,3,NaN,NaN
566317,IT1:41_983(1.0),A,107023,KILLINJ,F,2016,0,NaN,NaN
566328,IT1:41_983(1.0),A,107023,KILLINJ,M,2016,0,NaN,NaN


In [25]:
df_cities_grouped = df_combined[df_combined['Comune'].isnull()].groupby('REF_AREA').agg(count_unique_year=('TIME_PERIOD', 'nunique')).sort_values('REF_AREA', ascending=True)

print(df_cities_grouped)
print(df_cities_grouped['count_unique_year'].sum())

          count_unique_year
REF_AREA                   
6064                      1
6089                      1
13093                     1
13109                     1
13175                     1
...                     ...
107019                    1
107020                    1
107021                    1
107022                    1
107023                    1

[284 rows x 1 columns]
284


By looking at the city codes with null values and comparing the data frame grouped by code, we can see that there are 284 unique values, exactly 1/3 of the 852 rows in the entire data frame, and that the number of unique values in each year's column is always 1 (their sum is 284).   
Therefore, we can deduce with a high degree of certainty that for each "special" city in the INSTAT data, there are values for the year after they lost their independence. Therefore, the most conservative approach in this analysis is to eliminate the null values to avoid unintentionally including incorrect values and to rely solely on data from both the INSTAT and SITUAS datasets with which we have absolute certainty.

In [26]:
# drop NaN records
df_combined = df_combined[df_combined['Comune'].notnull()]

print(df_combined.info())
df_combined

<class 'pandas.core.frame.DataFrame'>
Index: 572700 entries, 0 to 573551
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATAFLOW     572700 non-null  object 
 1   FREQ         572700 non-null  object 
 2   REF_AREA     572700 non-null  int64  
 3   DATA_TYPE    572700 non-null  object 
 4   RESULT       572700 non-null  object 
 5   TIME_PERIOD  572700 non-null  int64  
 6   OBS_VALUE    572700 non-null  int64  
 7   Comune       572700 non-null  object 
 8   Residenti    572700 non-null  float64
dtypes: float64(1), int64(3), object(5)
memory usage: 43.7+ MB
None


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,Comune,Residenti
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,Agliè,2557.0
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,Agliè,2538.0
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,Agliè,2588.0
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,Agliè,2679.0
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,Agliè,2674.0
...,...,...,...,...,...,...,...,...,...
573547,IT1:41_983(1.0),A,111107,ROADACC,9,2020,2,Villaspeciosa,2549.0
573548,IT1:41_983(1.0),A,111107,ROADACC,9,2021,5,Villaspeciosa,2536.0
573549,IT1:41_983(1.0),A,111107,ROADACC,9,2022,1,Villaspeciosa,2575.0
573550,IT1:41_983(1.0),A,111107,ROADACC,9,2023,4,Villaspeciosa,2616.0


In [27]:
# Saving the joined dataframe
joined_data = 'data/insta_situas_traffic_accidents_data.csv'

if not os.path.exists(joined_data):
    df_combined.to_csv(joined_data, index=False)